# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

According to the `Materials and Methods` section of the paper, 17 marketing campaigns are represented by the data. These 17 campaigns ran between May 2008 and November 2010, and have a total of 79,354 contacts.

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [4]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



Several categorical columns (job, marital, education, default, housing, loan) use "unknown" as a value instead of a proper null. This functions as a missing value encoding rather than a real category, and needs a decision on whether to drop those rows or keep "unknown" as its own category.

The pdays column uses 999 as a placeholder for "not previously contacted." Left as is, a model would treat 999 as a real numeric value rather than a stand-in, skewing any distance-based or regression-based calculation. This needs either a recode or a separate flag column marking whether a client was previously contacted.

Most of the categorical columns (job, marital, education, default, housing, loan, contact, month, day_of_week, poutcome) need to be converted to numeric form before a classifier can use them, typically through one-hot encoding. Education is an exception, since its categories have a natural order and can be encoded ordinally instead. The target column y also needs conversion, from 'yes'/'no' text to 0/1.

The duration column is numeric already, but the data description notes it leaks the outcome, since a duration of 0 always corresponds to y='no', and duration isn't known until after a call ends. For a realistic predictive model, this column should be dropped rather than kept.

The remaining numeric columns appear fine and only need a dtype check after loading the data.

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

The business objective is to predict, before a call is made, whether a client is likely to subscribe to a term deposit, using client demographics, financial attributes, and prior campaign history. This allows the bank to prioritize outreach toward clients with a higher likelihood of subscribing, improving the efficiency of future marketing campaigns and reducing the cost of unproductive contacts.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

First, we must identify which features need to be changed. Since we are only working with the bank information features for now, we will only look at the `age`,`job`,`marital`,`education`,`default`,`housing`, and `loan` columns.

`Age` stays numerical as is, no transformation needed there.

`Job` and `marital` are nominal categories with no inherent order, so they get one-hot encoded. Each includes an "unknown" value, which just becomes its own column like any other category at this stage.

`Education` has a natural order (basic.4y through university.degree), so it's a candidate for ordinal encoding rather than one-hot.

`default`, `housing`, and `loan` are technically binary (yes/no) but include "unknown" as a third value, so they aren't purely binary. One-hot encoding handles this cleanly without forcing a false binary structure.

`y` is the target and gets mapped to 0/1.

In [6]:
# Only taking the bank information features
bank_features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
X = df[bank_features].copy()
y = df['y'].map({'no': 0, 'yes': 1})

# Ordinal encoding for education, since categories have a natural order
education_order = {
    'illiterate': 0,
    'basic.4y': 1,
    'basic.6y': 2,
    'basic.9y': 3,
    'high.school': 4,
    'professional.course': 5,
    'university.degree': 6,
    'unknown': -1  # flagged separately since it doesn't fit the order
}
X['education'] = X['education'].map(education_order)

# One-hot encoding for the remaining nominal categorical columns
nominal_cols = ['job', 'marital', 'default', 'housing', 'loan']
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [9]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_accuracy = baseline.score(X_test, y_test)
print(baseline_accuracy)

0.8875940762320952


The baseline performance comes from always predicting the majority class, since that sets the minimum bar any real model needs to beat.

Using a DummyClassifier with the most_frequent strategy, this baseline comes out to 0.8876. About 88.8% of clients in the dataset did not subscribe to a term deposit, so any of the four classifiers needs to clear roughly 88.76% accuracy to be considered better than guessing "no" every time.

This bar is easy to clear numerically but harder to clear in a meaningful way. A model could land around 89-90% accuracy by leaning toward "no" predictions while still missing most actual subscribers. Given the class imbalance and the goal of identifying which clients are worth contacting, accuracy alone should not be the deciding metric going forward. Recall or F1 score on the "yes" class will give a better read on whether a model is actually useful.

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

### Problem 9: Score the Model

What is the accuracy of your model?

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

##### Questions